[← 04 - Subqueries to CTEs](<04 - Subqueries to CTEs.ipynb>) · [Course Overview](<00 - Course Overview.ipynb>)

# 05 - Window Functions, Part 1

`GROUP BY` collapses rows into one summary row per group, you lose the detail. A window function computes something across a group of rows *without* collapsing them, you keep every row and get the aggregate alongside it.

> **By the end of this notebook you'll be able to:** number and deduplicate rows with `ROW_NUMBER()`, compare a row to its neighbor with `LAG()`/`LEAD()`, and compute a running total with `SUM() OVER (PARTITION BY ...)`.

![GROUP BY collapses rows. A window function doesn't.](graphics/05_groupby_vs_window.png)

<details>
<summary>Mermaid source</summary>

```mermaid
---
title: GROUP BY Collapses Rows. A Window Function Doesn't.
---
flowchart LR
    subgraph groupby [" 📦 GROUP BY: One Row Per Group "]
        direction TB
        g1(["CustomerID 1: 3 orders, $4,200 total"])
    end

    subgraph window [" 📋 Window Function: Every Row Kept "]
        direction TB
        w1(["Order 101, $1,000, running total $1,000"])
        w2(["Order 102, $1,700, running total $2,700"])
        w3(["Order 103, $1,500, running total $4,200"])
    end

    style groupby fill:#fef9c3,stroke:#eab308,stroke-width:1.5px,color:#713f12
    style window fill:#f0fdf4,stroke:#22c55e,stroke-width:1.5px,color:#14532d

    classDef groupNode fill:#fef9c3,stroke:#eab308,color:#713f12
    classDef windowNode fill:#dcfce7,stroke:#22c55e,color:#14532d

    class g1 groupNode
    class w1,w2,w3 windowNode
```

</details>

## 1. ROW_NUMBER(): numbering and deduplicating rows

`ROW_NUMBER()` assigns a sequential number within each `PARTITION BY` group, in the order given by `ORDER BY`. A common use: finding the single most recent order per customer.

**Example:**

In [ ]:
WITH numbered_orders AS (
    SELECT
        CustomerID,
        SalesOrderID,
        OrderDate,
        ROW_NUMBER() OVER (PARTITION BY CustomerID ORDER BY OrderDate DESC) AS "Order Rank"
    FROM Sales.SalesOrderHeader
)
SELECT CustomerID, SalesOrderID, OrderDate
FROM numbered_orders
WHERE "Order Rank" = 1;
-- Order Rank resets to 1 for every new CustomerID, so this keeps only each customer's most recent order

## 2. LAG() and LEAD(): comparing a row to its neighbor

`LAG()` looks back at a previous row, `LEAD()` looks ahead, both within the same `PARTITION BY` group, without a self-join. Here we compare each of a customer's orders to the one before it:

**Example:**

In [ ]:
SELECT
    CustomerID,
    SalesOrderID,
    OrderDate,
    TotalDue,
    LAG(TotalDue) OVER (PARTITION BY CustomerID ORDER BY OrderDate) AS "Previous Order Total"
FROM Sales.SalesOrderHeader
ORDER BY CustomerID, OrderDate;

The first order for each customer has no previous order, so `LAG()` returns `NULL` there, exactly as it should.

## 3. SUM() OVER (PARTITION BY ...): a running total without losing row detail

This is the running-total example from the diagram at the top of this notebook, made real:

**Example:**

In [ ]:
SELECT
    CustomerID,
    SalesOrderID,
    OrderDate,
    TotalDue,
    SUM(TotalDue) OVER (PARTITION BY CustomerID ORDER BY OrderDate) AS "Running Total"
FROM Sales.SalesOrderHeader
ORDER BY CustomerID, OrderDate;
-- Running Total accumulates within each CustomerID, and resets when CustomerID changes

## A gotcha worth knowing now

Window functions run after `WHERE` and `GROUP BY`, but before `ORDER BY`, in SQL Server's actual order of execution (the same order-of-execution idea from `01 - SQL Basics`). That's why you can't reference a window function's alias inside the same query's `WHERE` clause, `WHERE` has already run by the time the window function's result exists. If you need to filter on a window function's result, wrap it in a CTE first, the way `numbered_orders` did above.

## What's Next

`06 - Window Functions, Part 2` covers frames (`ROWS BETWEEN ...`), `NTILE()`, `PERCENT_RANK()`, `CUME_DIST()`, and the `LAST_VALUE()` frame trap, marked "Coming Soon" in the Course Agenda until it's written.

---

[← 04 - Subqueries to CTEs](<04 - Subqueries to CTEs.ipynb>) · [Course Overview](<00 - Course Overview.ipynb>)

*SQL_Tutorial* is written and maintained by Samuel Shaibu as part of *All About Data & More*. Licensed under [MIT](LICENSE).